# ⚡ 1-Click Hardware WebGPU Chrome on Google Colab (Tesla T4) — 60 FPS WebRTC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hosein-ul/colab-webgpu-chrome/blob/main/colab_chrome_webgpu.ipynb)

This notebook configures a **Google Colab NVIDIA Tesla T4** instance with full hardware-accelerated **WebGPU**, launching official Google Chrome streamed at **60 FPS crystal-clear WebRTC with sub-30ms latency** via **Google Chrome Remote Desktop (CRD)**.

---
### 🚀 Quick Start / راهنمای اجرای سریع:
1. Make sure your runtime is set to **T4 GPU**: (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU` ➔ `Save`).
2. Open **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** in a new tab, click **Begin ➔ Next ➔ Authorize**, and copy the command for **Debian Linux** (starts with `DISPLAY= /opt/google/...`).
3. Paste the command into `AUTH_COMMAND` below, choose any 6-digit PIN (default `123456`), and click the **Play** button.
4. Everything configures autonomously in ~40 seconds! Click the green button to access your 60 FPS remote desktop at **[remotedesktop.google.com/access](https://remotedesktop.google.com/access)**.

> 🔒 **Note for Private Repos in Colab:** When opening via GitHub in Colab, make sure to check **"Include private repositories"** in the GitHub tab so Colab can load your private notebook.

In [ ]:
#@title 🚀 1-Click Chrome Remote Desktop Launcher (Tesla T4 WebGPU + 60 FPS WebRTC)
#@markdown ### 📋 Authentication Instructions:
#@markdown 1. Visit **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)**
#@markdown 2. Click **Begin** ➔ **Next** ➔ **Authorize**
#@markdown 3. Copy the **Debian Linux** command and paste it below:

AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}

import os, sys, time, subprocess, re
from IPython.display import display, HTML

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 لطفاً وارد لینک شوید: https://remotedesktop.google.com/headless")
    print("سپس دکمه‌های Begin -> Next -> Authorize را بزنید و کد داده‌شده را کپی کنید.")
    raw_auth = input("دستور احراز هویت (AUTH_COMMAND) را اینجا پیست کنید: ").strip()

if not raw_auth:
    raise ValueError("دستور احراز هویت وارد نشد. لطفاً سلول را دوباره اجرا کنید.")

print("⏳ [1/5] پیکربندی درایورهای سطح هسته NVIDIA Tesla T4 و Vulkan...")
os.makedirs("/dev/dri", exist_ok=True)
subprocess.run("mknod -m 666 /dev/dri/card0 c 226 0 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/dri/renderD128 c 226 128 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/nvidia-modeset c 195 254 > /dev/null 2>&1", shell=True)
subprocess.run("chmod -R 666 /dev/dri /dev/nvidia* > /dev/null 2>&1", shell=True)

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd_content = '{\n    "file_format_version": "1.0.0",\n    "ICD": {\n        "library_path": "libGLX_nvidia.so.0",\n        "api_version": "1.3.275"\n    }\n}'
for p in ["/etc/vulkan/icd.d/nvidia_icd.json", "/usr/share/vulkan/icd.d/nvidia_icd.json"]:
    with open(p, "w") as f: f.write(icd_content)

with open("/etc/ld.so.conf.d/nvidia.conf", "w") as f:
    f.write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

print("⏳ [2/5] ایجاد کاربر اختصاصی و پیکربندی مجوزهای امنیتی...")
os.makedirs("/home/colab", exist_ok=True)
subprocess.run("id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1", shell=True)
subprocess.run("echo 'colab:123456' | chpasswd", shell=True)
subprocess.run("usermod -aG sudo,video,render colab >/dev/null 2>&1", shell=True)
with open("/etc/sudoers.d/colab", "w") as f:
    f.write("colab ALL=(ALL) NOPASSWD:ALL\n")

print("⏳ [3/5] به‌روزرسانی پکیج‌ها و نصب XFCE4 و Chrome Remote Desktop...")
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get update -qq", shell=True)
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal xscreensaver > /dev/null 2>&1", shell=True)
subprocess.run("systemctl disable lightdm.service > /dev/null 2>&1 || true", shell=True)

crd_deb = "/tmp/crd.deb"
subprocess.run(f"wget -q -O {crd_deb} https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb", shell=True)
subprocess.run(f"dpkg --install {crd_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
subprocess.run(f"rm -f {crd_deb}", shell=True)
subprocess.run("usermod -aG chrome-remote-desktop colab >/dev/null 2>&1", shell=True)

crd_session = "exec /etc/X11/Xsession /usr/bin/xfce4-session\n"
with open("/etc/chrome-remote-desktop-session", "w") as f: f.write(crd_session)
with open("/home/colab/.chrome-remote-desktop-session", "w") as f: f.write(crd_session)

print("⏳ [4/5] نصب Google Chrome و تنظیم اجرای خودکار WebGPU...")
if not os.path.exists("/usr/bin/google-chrome"):
    chrome_deb = "/tmp/chrome.deb"
    subprocess.run(f"wget -q -O {chrome_deb} https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb", shell=True)
    subprocess.run(f"dpkg -i {chrome_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
    subprocess.run(f"rm -f {chrome_deb}", shell=True)

launch_script = f"""#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-vulkan=native \\
  --use-angle=vulkan \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --disable-background-timer-throttling \\
  --disable-backgrounding-occluded-windows \\
  --disable-renderer-backgrounding \\
  --start-maximized \\
  "$@" \\
  "{TARGET_URL}" &
"""
with open("/home/colab/launch_chrome.sh", "w") as f: f.write(launch_script)
subprocess.run("chmod +x /home/colab/launch_chrome.sh", shell=True)

os.makedirs("/home/colab/.config/autostart", exist_ok=True)
os.makedirs("/home/colab/Desktop", exist_ok=True)
desktop_entry = """[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Comment=Google Chrome with Hardware WebGPU on Tesla T4
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Path=/home/colab
Terminal=false
StartupNotify=true
"""
with open("/home/colab/.config/autostart/webgpu-chrome.desktop", "w") as f: f.write(desktop_entry)
with open("/home/colab/Desktop/WebGPU_Chrome.desktop", "w") as f: f.write(desktop_entry)
subprocess.run("chmod +x /home/colab/Desktop/WebGPU_Chrome.desktop", shell=True)
subprocess.run("chown -R colab:colab /home/colab && chmod +x /home/colab/.chrome-remote-desktop-session", shell=True)

print("⏳ [5/5] راه‌اندازی هاست Chrome Remote Desktop با اکانت شما...")
if raw_auth.startswith("4/"):
    exec_cmd = f'/opt/google/chrome-remote-desktop/start-host --code="{raw_auth}" --redirect-url="https://remotedesktop.google.com/_/oauthredirect" --name=colab-t4'
else:
    exec_cmd = raw_auth

pin_str = str(PIN)
if "--pin=" not in exec_cmd:
    exec_cmd += f" --pin={pin_str}"

exec_cmd = re.sub(r'^DISPLAY=\s*', '', exec_cmd)
full_cmd = f'su - colab -c \'{exec_cmd}\''

proc = subprocess.Popen(
    full_cmd,
    shell=True,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
try:
    stdout, stderr = proc.communicate(input=f"{pin_str}\n{pin_str}\n", timeout=25)
except subprocess.TimeoutExpired:
    proc.kill()
    stdout, stderr = proc.communicate()

time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop", shell=True, capture_output=True, text=True)

print("\n" + "═"*60)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Chrome Remote Desktop با موفقیت راه‌اندازی شد!")
    print(f"🔑 پین ورود شما: {pin_str}")
    print("═"*60)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری آماده اتصال است! (WebRTC 60 FPS)</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                سیستم <b>colab-t4</b> در پنل گوگل اضافه شد. روی دکمه زیر کلیک کرده و پین <b>{pin_str}</b> را وارد نمایید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    '''))
else:
    print("⚠️ پیام خروجی:")
    print(stdout)
    if stderr: print("خطاها:", stderr)
    print("═"*60)


In [ ]:
#@title 📊 مانیتور زنده هش‌ریت و مصرف کارت گرافیک تسلا T4 (Real-Time GPU Monitor)
#@markdown وضعیت زنده درصد درگیری GPU، حافظه VRAM، توان مصرفی و دمای کارت گرافیک:

import time, subprocess
from IPython.display import clear_output

print("Starting lightweight GPU monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*60)
            print(f"🚀 UNICRED WEBGPU MINING MONITOR — {name}")
            print("="*60)
            print(f"  GPU Compute Utilization : {gpu_util}%")
            print(f"  VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  GPU Temperature         : {temp} °C")
            print(f"  Power Consumption       : {power} W")
            print("="*60)
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
